In [1]:
import pandas as pd

In [8]:
# 读取数据文件，这里假设文件和代码在同一目录下，若不在需指定完整路径
data = pd.read_csv('13.nino4.long.anom.data.csv', delim_whitespace=True, header=None)

# 设置列名
columns = ['Year'] + [f'{month}' for month in ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']]
data.columns = columns


# 融化数据，将宽格式转换为长格式，抽取时间信息
data_melted = pd.melt(data, id_vars='Year', var_name='Month', value_name='Temperature_Anomaly')

# 组合年份和月份为Date列
data_melted['Date'] = pd.to_datetime(data_melted['Year'].astype(str) + '-' + data_melted['Month'])

# 选择需要的列
result = data_melted[['Date', 'Temperature_Anomaly']]

print(result)


df = pd.read_csv('nino3_classified.csv')
lanina_starts, nino_starts = [], []
current_label, start_date, count = None, None, 0

for idx, row in df.iterrows():
    label = row['Label']
    if label == 'LaNinaTemp':
        if current_label == 'LaNinaTemp':
            count += 1
        else:
            current_label = 'LaNinaTemp'
            count = 1
            start_date = row['Date']
        if count >= 6:
            lanina_starts.append(start_date)
            count = 0 # 重置计数，避免重复记录
    elif label == 'NinoTemp':
        if current_label == 'NinoTemp':
            count += 1
        else:
            current_label = 'NinoTemp'
            count = 1
            start_date = row['Date']
        if count >= 6:
            nino_starts.append(start_date)
            count = 0 # 重置计数
        else:
            current_label, start_date, count = None, None, 0 # 非异常状态重置

# 保存结果
with open('LaNinaStartDates.txt', 'w') as f:
    f.write('\n'.join(lanina_starts))

with open('NinoStartDates.txt', 'w') as f:
    f.write('\n'.join(nino_starts))

ValueError: Length mismatch: Expected axis has 1 elements, new values have 13 elements